# Nuclide Build Up During Depletion Simulation

This example simulates the build up of activate products within a material under neutron irradiation. The subsequent decay of unstable isotopes is also simulated.

This first cell imports the packages needed, note the extra import openmc.deplete import

In [1]:
# remove any old files
!rm settings.xm model.xml materials.xml geometry.xml settings.xml

import openmc
import openmc.deplete
from pathlib import Path
# Setting the cross section path to the correct location in the docker image.
# If you are running this outside the docker image you will have to change this path to your local cross section path.
openmc.config['cross_sections'] = Path.home() / 'nuclear_data' / 'cross_sections.xml'
# the chain file was downloaded with
# pip install openmc_data
# download_endf_chain -r b8.0
openmc.config['chain_file'] = Path.home() / 'nuclear_data' / 'chain-endf-b8.0.xml'


rm: cannot remove 'settings.xm': No such file or directory
rm: cannot remove 'model.xml': No such file or directory
rm: cannot remove 'materials.xml': No such file or directory
rm: cannot remove 'geometry.xml': No such file or directory
rm: cannot remove 'settings.xml': No such file or directory


This section creates the geometry and the cells.
Note that it it necessary to set the volume of the material or cell.
This is so that the depletion code can find the number of atoms within the cell given the material composition, material density and volume.

In [2]:

import math

# MATERIALS

# makes a simple material from Silver
my_material = openmc.Material() 
my_material.add_element('Ag', 1, percent_type='ao')
my_material.set_density('g/cm3', 10.49)


sphere_radius = 100
volume_of_sphere = (4/3) * math.pi * math.pow(sphere_radius, 3)
my_material.volume = volume_of_sphere  # a volume is needed so openmc can find the number of atoms in the cell/material
my_material.depletable = True  # depletable = True is needed to tell openmc to update the material with each time step

materials = openmc.Materials([my_material])
materials.export_to_xml()


# GEOMETRY

# surfaces
sph1 = openmc.Sphere(r=sphere_radius, boundary_type='vacuum')

# cells, makes a simple sphere cell
shield_cell = openmc.Cell(region=-sph1)
shield_cell.fill = my_material

# sets the geometry to the universe that contains just the one cell
geometry = openmc.Geometry([shield_cell])



This section defines the neutron source term to use and the settings

In [3]:
# creates a 14MeV neutron point source
source = openmc.IndependentSource()
source.space = openmc.stats.Point((0, 0, 0))
source.angle = openmc.stats.Isotropic()
source.energy = openmc.stats.Discrete([14e6], [1])
source.particles = 'neutron'

# SETTINGS

# Instantiate a Settings object
settings = openmc.Settings()
settings.batches = 2
settings.inactive = 0
settings.particles = 10000
settings.source = source
settings.run_mode = 'fixed source'

model = openmc.model.Model(geometry, materials, settings)

This is the depletion specific part of the model setup.

This section specifies the chain file, this tells openmc the decay paths between isotopes including probabilities of different routes and half lives

This next stage sets the time steps and corresponding source rates for the irradiation schedule.

An output file will be produced with showing the material composition at every time step.

We are irradiating the Silver for multiple half lives to show build up and saturation

Saturation happens when decay is = to creation of the particular isotope

Ag110 half life is 24 seconds so it will start to become saturated after 120 seconds

Ag108 half life is 145 seconds so it will not be saturated

In [4]:
# We define timesteps together with the source rate to make it clearer
timesteps_and_source_rates = [
    (24, 1e20),
    (24, 1e20),
    (24, 1e20),
    (24, 1e20),
    (24, 1e20),  # should saturate Ag110 here as it has been irradiated for over 5 halflives
    (24, 1e20),
    (24, 1e20),
    (24, 1e20),
    (24, 1e20),
    (24, 0),
    (24, 0),
    (24, 0),
    (24, 0),
    (24, 0),
    (24, 0),
    (24, 0),
    (24, 0),
    (24, 0),
    (24, 0),
    (24, 0),
]

# Uses list Python comprehension to get the timesteps and source_rates separately
timesteps = [item[0] for item in timesteps_and_source_rates]
source_rates = [item[1] for item in timesteps_and_source_rates]


# PredictorIntegrator has been selected as the depletion operator for this example as it is a fast first order Integrator
# OpenMC offers several time-integration algorithms https://docs.openmc.org/en/stable/pythonapi/deplete.html#primary-api\n",
# CF4Integrator should normally be selected as it appears to be the most accurate https://dspace.mit.edu/handle/1721.1/113721\n",

model.deplete(
    timesteps,
    source_rates=source_rates,
    method="predictor",  # predictor is a simple but quick method
    operator_kwargs={
        "normalization_mode": "source-rate",  # needed as this is a fixed source simulation
        "chain_file": openmc.config['chain_file'],
        "reduce_chain_level": 5,
    },
)

                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################

 Reading materials XML file...
 Reading geometry XML file...
 Reading Ag107 from /home/jon/nuclear_data/neutron/Ag107.h5
 Reading Ag109 from /home/jon/nuclear_data/neutron/Ag109.h5
 Minimum neutron data temperature: 294.0 K
 Maximum neutron data temperature: 294.0 K
 Reading tallies XML file...
 Preparing distributed cell instances...
 Reading plot XML file...
 Writing summary.h5 file...


[openmc.deplete] t=0.0 s, dt=24 s, source=1e+20
 Reading H1 from /home/jon/nuclear_data/neutron/H1.h5
 Reading H2 from /home/jon/nuclear_data/neutron/H2.h5
 Reading H3 from /home/jon/nuclear_data/neutron/H3.h5
 Reading He3 from /home/jon/nuclear_data/neutron/He3.h5
 Reading He4 from /home/jon/nuclear_data/neutron/He4.h5
 Reading Br81 from /home/jon/nuclear_data/neutron/Br81.h5
 Reading Kr83 from /home/jon/nuclear_data/neutron/Kr83.h5


 Reading Kr84 from /home/jon/nuclear_data/neutron/Kr84.h5
 Reading Kr85 from /home/jon/nuclear_data/neutron/Kr85.h5
 Reading Kr86 from /home/jon/nuclear_data/neutron/Kr86.h5


 Reading Rb85 from /home/jon/nuclear_data/neutron/Rb85.h5
 Reading Rb86 from /home/jon/nuclear_data/neutron/Rb86.h5
 Reading Rb87 from /home/jon/nuclear_data/neutron/Rb87.h5


...

[openmc.deplete] t=384.0 s, dt=24 s, source=0.0
 Creating state point openmc_simulation_n16.h5...


[openmc.deplete] t=408.0 s, dt=24 s, source=0.0
 Creating state point openmc_simulation_n17.h5...


[openmc.deplete] t=432.0 s, dt=24 s, source=0.0
 Creating state point openmc_simulation_n18.h5...


[openmc.deplete] t=456.0 s, dt=24 s, source=0.0
 Creating state point openmc_simulation_n19.h5...


[openmc.deplete] t=480.0 (final operator evaluation)
 Creating state point openmc_simulation_n20.h5...


This next section starts the depletion simulation and produces the output files

This section extracts the results of the depletion simulation from the h5 file and gets the amount of Ag110 in the material at each of the time steps

In [5]:
results = openmc.deplete.ResultsList.from_hdf5("depletion_results.h5")

times, number_of_Ag110_atoms = results.get_atoms(my_material, 'Ag110')

for time, num in zip(times, number_of_Ag110_atoms):
    print(f" Time {time}s. Number of Ag110 atoms {num}")

/home/jon/.neutronicsworkshop/lib/python3.11/site-packages/openmc/deplete/results.py:94: FutureWarning: The ResultsList.from_hdf5(...) method is no longer necessary and will be removed in a future version of OpenMC. Use Results(...) instead.
  warn(


 Time 0.0s. Number of Ag110 atoms 0.0
 Time 24.0s. Number of Ag110 atoms 1.390899392955125e+21
 Time 48.0s. Number of Ag110 atoms 2.0976170342669038e+21
 Time 72.0s. Number of Ag110 atoms 2.4609439301756566e+21
 Time 96.0s. Number of Ag110 atoms 2.6447562029332436e+21
 Time 120.0s. Number of Ag110 atoms 2.7394373458056367e+21
 Time 144.0s. Number of Ag110 atoms 2.793995316128349e+21
 Time 168.0s. Number of Ag110 atoms 2.8018322869104447e+21
 Time 192.0s. Number of Ag110 atoms 2.8076380625555617e+21
 Time 216.0s. Number of Ag110 atoms 2.816110462829026e+21
 Time 240.0s. Number of Ag110 atoms 1.4320622197003313e+21
 Time 264.0s. Number of Ag110 atoms 7.282392637144813e+20
 Time 288.0s. Number of Ag110 atoms 3.703277849549926e+20
 Time 312.0s. Number of Ag110 atoms 1.883208959520983e+20
 Time 336.0s. Number of Ag110 atoms 9.576586597840663e+19
 Time 360.0s. Number of Ag110 atoms 4.869933118754419e+19
 Time 384.0s. Number of Ag110 atoms 2.4764827597092327e+19
 Time 408.0s. Number of Ag110 

In addition to Ag110 other atoms get created. This section plots the number of nuclides in the material excluding the original nuclides in the unirradiated material

In [6]:
import openmc_depletion_plotter
# this package provides convenient plotting methods for depletion simulations like this one
# more details here https://github.com/fusion-energy/openmc_depletion_plotter

results.plot_atoms_vs_time(excluded_material=my_material)

/home/jon/.neutronicsworkshop/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Material instance already exists with id=1.
  warn(msg, IDWarning)


Not all nuclide are unstable and the unstable ones have a different half life. This next plot shows the specific activity (activity per unit mass) as a function of time.

This is useful for identifying a suitable waste repository for activated waste.

In [7]:
results.plot_activity_vs_time()

/home/jon/.neutronicsworkshop/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning:

Another Material instance already exists with id=1.

